In [ ]:
import pyodbc, struct, json as _json, traceback as _tb

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"

def write_diag(name, obj):
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}/Files/diagnostics/{name}.json"
    notebookutils.fs.put(path, _json.dumps(obj, indent=2, default=str), overwrite=True)
    print(f"wrote {path}")

try:
    server = "4oa7g2g2i24uposxn4zcxdynue-y2u3efqymrpurbs7yr2efofwyy.datawarehouse.fabric.microsoft.com"
    database = "wh_gold"
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};Encrypt=yes;"
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
    cursor = conn.cursor()
    print("connected")
except Exception as e:
    write_diag("meta_health_connect", {"status": "failed", "error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
import time

try:
    # dbo, not a custom "meta" schema -- every other table in wh_gold
    # lives in dbo, and a custom schema turned out to make Direct Lake's
    # table creation hang indefinitely rather than just take the usual
    # extra discovery time (see ADR 0020). "load_log" as a name is
    # distinctive enough on its own without a separate schema.
    # One-time cleanup of the abandoned first attempt.
    cursor.execute("SELECT COUNT(*) FROM sys.schemas WHERE name = 'meta'")
    if cursor.fetchone()[0] == 1:
        cursor.execute("SELECT COUNT(*) FROM sys.tables t JOIN sys.schemas s ON s.schema_id = t.schema_id WHERE s.name = 'meta' AND t.name = 'load_log'")
        if cursor.fetchone()[0] == 1:
            cursor.execute("DROP TABLE meta.load_log")
        cursor.execute("DROP SCHEMA meta")

    def wait_for_table(name, attempts=10):
        for _ in range(attempts):
            cursor.execute("SELECT COUNT(*) FROM sys.tables WHERE name = ?", name)
            if cursor.fetchone()[0] == 1:
                return
            time.sleep(2)
        raise RuntimeError(f"dbo.{name} not visible in sys.tables after {attempts} retries")

    cursor.execute("SELECT COUNT(*) FROM sys.tables WHERE name = 'load_log'")
    if cursor.fetchone()[0] == 0:
        cursor.execute("""
            CREATE TABLE dbo.load_log (
                source VARCHAR(20) NOT NULL,
                table_name VARCHAR(100) NOT NULL,
                row_count BIGINT NOT NULL,
                checked_at_utc DATETIME2(6) NOT NULL,
                status VARCHAR(20) NOT NULL,
                notes VARCHAR(500) NULL
            )
        """)
        wait_for_table("load_log")
    print("dbo.load_log ready")
except Exception as e:
    write_diag("meta_health_create_ERROR", {"error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
try:
    import datetime
    checked_at = datetime.datetime.utcnow()
    tables = [('dim_date', 'derived'), ('dim_time_15', 'derived'), ('dim_airport', 'reference'), ('dim_carrier', 'BTS'), ('dim_aircraft', 'FAA'), ('dim_delay_cause', 'reference'), ('dim_cancel_reason', 'reference'), ('dim_bank', 'derived'), ('fact_flight_leg', 'BTS'), ('fact_aircraft_turn', 'BTS'), ('fact_station_hour', 'BTS'), ('fact_checkpoint_hour', 'TSA'), ('fact_cbp_wait_hour', 'CBP'), ('fact_rebank_effect', 'analysis'), ('fact_rebank_event_study', 'analysis'), ('fact_station_day_scorecard', 'analysis'), ('fact_tsa_departure_hour_profile', 'TSA'), ('fact_cbp_hour_profile', 'CBP')]
    rows = []
    for table_name, source in tables:
        cursor.execute(f"SELECT COUNT(*) FROM sys.tables WHERE name = ?", table_name)
        exists = cursor.fetchone()[0] == 1
        if not exists:
            rows.append((source, table_name, 0, checked_at, "MISSING", "table not found in wh_gold"))
            continue
        cursor.execute(f"SELECT COUNT(*) FROM dbo.[{table_name}]")
        n = cursor.fetchone()[0]
        status = "OK" if n > 0 else "EMPTY"
        rows.append((source, table_name, n, checked_at, status, None))

    # Point-in-time reconciliation note (see docs/kpi_spec.md's
    # "Reconciliation target" and ADR 0013) -- not recomputed live here,
    # this sweep is a freshness/row-count check, not a full 44-month
    # bronze-vs-gold aggregate reconciliation (expensive, already done
    # once and confirmed on 2026-10-31).
    rows.append((
        "reconciliation", "fact_flight_leg (vs bronze)", 27787225, checked_at, "OK",
        "Monthly D0/A14/CF, gold vs lh_bronze.bts_marketing_ontime raw aggregates: "
        "max D0 diff 0.0000924, max CF diff 0.0000087 across all 44 months (2026-10-31 full build). "
        "See ADR 0013 and docs/kpi_spec.md.",
    ))

    cursor.fast_executemany = True
    cursor.executemany(
        "INSERT INTO dbo.load_log (source, table_name, row_count, checked_at_utc, status, notes) VALUES (?, ?, ?, ?, ?, ?)",
        rows,
    )

    result = {"checked_at_utc": str(checked_at), "tables_swept": len(rows), "not_ok": [r[1] for r in rows if r[4] != "OK"]}
    print(_json.dumps(result, indent=2))
    write_diag("meta_health_sweep", result)
except Exception as e:
    write_diag("meta_health_sweep_ERROR", {"error": repr(e), "traceback": _tb.format_exc()})
    raise
